<a href="https://colab.research.google.com/github/kitlapp/NLP-Course/blob/main/pr_042_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 4.2 Retreival Aubmented Generation - RAG - for Conversational AI


This notebook implements a Retrieval-Augmented Generation (RAG) pipeline for conversational AI by adapting the drug reviews dataset for similarity-based question-answering.  

 It builds a vector database using dense sentence embeddings through the pretrained HF model 'BAAI/bge-small-en-v1.5' and FAISS as a vector database to store the embeddings alongside with an index to perform similarity searches.

 A pre-trained causal language model "Qwen/Qwen3.5-0.8B" is used to generate  responses from the most relevant embedding retrieved.

 Prompt engineering guidelines were utilized in order to set the guard rails on the usage of metadata, ratings, and specific side-effect fields.   

 Finally, the notebook evaluates how well the RAG system synthesizes the responses and addresses retrieval sample biases against ground-truth data derived from the dataframe itself through Pandas.

 From this notebook the following files are produced and saved in shared Google Drive folder "nip_project_data":



*   Embeddings of reviews : review_embeddings_\<*model*>.npy
*   FAISS Indexes of the review embeddings : review_index_\<*model*>.faiss
*   The dataset created by RAGAS with the data to be analyzed : directory my_rag_eval_dataset
*   The excel file with the RAGAS evaluation results : rag_evaluation_results.xlsx



# Import Libraries

In [1]:
#Vector Database to use instead of Cosine Similarity
!pip install faiss-cpu

In [ ]:
#For RAGAS Evaluation
#!pip install langchain-google-vertexai
!pip install ragas datasets
!pip install "langchain<1.0" "langchain-community<1.0" "langchain-openai<1.0" --no-cache-dir

In [ ]:
!pip install python-dotenv

In [36]:
# Standard library
from pathlib import Path
import re
import os
import shutil
from dotenv import load_dotenv

# Third-party libraries
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

#For RAG
from sentence_transformers import SentenceTransformer
import torch
import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, TextStreamer, GenerationConfig
#
import faiss #Vector Database
from huggingface_hub.inference._generated.types import question_answering

pd.set_option('display.max_colwidth', None)

In [72]:
#For RAGAS Evaluation
from datasets import Dataset
from datasets import load_from_disk
#
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import faithfulness, answer_relevancy
from ragas import evaluate

# Import files from Kaggle Hub


We are reloading from Kaggle as we do not need all this preprocessing done for traditional classification models. Here we are going to use the "Reviews" Column as is, performing only the necessary steps to maintain the natural language flavor that LLMs need.  

LLMs need natural language to understand the intent and sentiment. If we strip  punctuation or stop words or lematize or tokenize we will make the text "robotic" and harder for the model to interpret. So, we leave as is.

In [38]:
# Define dataset ID according to KaggleHub format
path_to_dataset = "rohanharode07/webmd-drug-reviews-dataset"

# Download the latest available version of the dataset
# If the same version is already cached locally, it will not download again
dataset_path = kagglehub.dataset_download(path_to_dataset)

print("Dataset location:", dataset_path)

# Convert the returned path into a Path object for easier handling later
dataset_path = Path(dataset_path)

# Iterate over all items inside the dataset directory (including version folder)
print("Files found:")
for file in dataset_path.iterdir():
    print(file.name)

# Use the Path object to create the final path to the dataset
raw_filepath = dataset_path / "webmd.csv"
print("Path to raw dataset:", raw_filepath)

# Read it to a df
df = pd.read_csv(raw_filepath)

Dataset location: /root/.cache/kagglehub/datasets/rohanharode07/webmd-drug-reviews-dataset/versions/1
Files found:
webmd.csv
Path to raw dataset: /root/.cache/kagglehub/datasets/rohanharode07/webmd-drug-reviews-dataset/versions/1/webmd.csv


# Paths for common folders

In [39]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [40]:
# Path to root shared data folder (should work for every user after adding the shortcut)
ROOT_DATA_DIR = Path("/content/drive/MyDrive/nlp_project_data")

# Check that the folder exists
if ROOT_DATA_DIR.exists():
    print(f"Shared dataset folder found: {ROOT_DATA_DIR}")
else:
    raise FileNotFoundError(
        f"Dataset folder not found: {ROOT_DATA_DIR}\n"
        "Read the README instructions or the text cells in the sections "
        "'Mount Google Drive' and 'Handle Paths'.\n"
        "Ensure that the shared dataset folder has been added as a shortcut "
        "inside your My Drive, then rerun this cell."
    )

Shared dataset folder found: /content/drive/MyDrive/nlp_project_data


In [41]:
os.chdir('/content/drive/MyDrive/MSc NLP/NaturalLanguageProcessing-main/NaturalLanguageProcessing-main')
# Verify the move
print("Current directory:", os.getcwd())

Current directory: /content/drive/MyDrive/MSc NLP/NaturalLanguageProcessing-main/NaturalLanguageProcessing-main


OPEN_API_KEY environmental variable

In [42]:
load_dotenv()

True

# Specific preprocessing steps from pr_01_data_collection_and_preprocessing

The following function integrates specific data integrity checks from our preliminary preprocessing (pr_01_data_collection_and_preprocessing), omitting steps like lowercasing, lemmatization, stop-word and emoji/emoticon removal so as to allow for grammatical structure and sentiment.

In [46]:
null_like_values = {"none", "nan", "null", "n/a", "na"}

def clean_data_for_rag(df, max_repeat=2):
    df = df.copy()
    print(f"Starting rows: {len(df)}")

    df = df.dropna(subset=['Reviews'])
    #print(f"1. After dropna: {len(df)}")
    df['Reviews'] = df['Reviews'].astype(str).str.strip()
    #print(f"2. After strip: {len(df)}")
    #print("Duplicates:", df.duplicated().sum())
    df = df.drop_duplicates()
    #print(f"8. After final deduplication: {len(df)}")

    score_cols = ['Satisfaction', 'Effectiveness', 'EaseofUse']
    for col in score_cols:
        if col in df.columns:
            df = df[df[col] <= 5]
    #print(f"3. After score filters (>5): {len(df)}")

    df = df[~df['Reviews'].str.lower().isin(null_like_values)]
    #print(f"4. After null-like string filter: {len(df)}")

    df = df[df['Reviews'] != ""]
    df = df[df['Reviews'].str.len() > 1]
    #print(f"5. After empty and 1-char filter: {len(df)}")

    def is_punct_only(text):
        return bool(re.fullmatch(r"[\W_]+", text))
    df = df[~df['Reviews'].apply(is_punct_only)]
    #print(f"6. After punctuation-only filter: {len(df)}")

    def is_repeated_single_char(text):
        return bool(re.fullmatch(r"(.)\1+", text.lower()))
    df = df[~df['Reviews'].apply(is_repeated_single_char)]
    #print(f"7. After repeated single-char filter: {len(df)}")

    def sanitize(text):
        pattern = r'(\w)\1{' + str(max_repeat) + r',}'
        text = re.sub(pattern, r'\1' * max_repeat, text)
        text = re.sub(r'http\S+|www\S+', '[URL]', text)
        text = re.sub(r'@\w+', '[USER]', text)
        return text

    df['Reviews'] = df['Reviews'].apply(sanitize)
    print(f"Final rows: {len(df)}")


    return df

# Run the diagnostic
df_rag = clean_data_for_rag(df)

Starting rows: 362806
Final rows: 319933


In [8]:
# Verification prints
print(f"Final RAG dataset dimensionality: {df_rag.shape}\n")
print("Final RAG dataset first 2 rows:")
df_rag.head()

Final RAG dataset dimensionality: (319933, 12)

Final RAG dataset first 2 rows:


,Age,Condition,Date,Drug,DrugId,EaseofUse,Effectiveness,Reviews,Satisfaction,Sex,Sides,UsefulCount
0,75 or over,Stuffy Nose,9/21/2014,25dph-7.5peh,146724,5,5,I'm a retired physician and of all the meds I have tried for my allergies (seasonal and not) - this one is the most effective for me. When I first began using this drug some years ago - tiredness as a problem but is not currently.,5,Male,"Drowsiness, dizziness , dry mouth /nose/throat, headache , upset stomach , constipation , or trouble sleeping may occur.",0
1,25-34,Cold Symptoms,1/13/2011,25dph-7.5peh,146724,5,5,cleared me right up even with my throat hurting it went away after taking the medicine,5,Female,"Drowsiness, dizziness , dry mouth /nose/throat, headache , upset stomach , constipation , or trouble sleeping may occur.",1
2,65-74,Other,7/16/2012,warfarin (bulk) 100 % powder,144731,2,3,why did my PTINR go from a normal of 2.5 to over \n100?,3,Female,,0
3,75 or over,Other,9/23/2010,warfarin (bulk) 100 % powder,144731,2,2,FALLING AND DON'T REALISE IT,1,Female,,0
4,35-44,Other,1/6/2009,warfarin (bulk) 100 % powder,144731,1,1,"My grandfather was prescribed this medication (Coumadin) to assist in blood thinning due to a heart and thyroid condition. His primary doctor was aware that he was on an aspirin regiment and still prescribed this medicine, it caused his blood to thin out to much and he ended up internally bleeding to death. If you are going to take this medicine please ask your doctors about possible side effects or drug interactions.",1,Male,,1


## RAG Preprocessing steps. Which steps we actually used and which we did not

| Preprocessing Step (ref. pr_01) | Action | Justification for RAG |
| :--- | :--- | :--- |
| **2.6.1 - 2.6.5** (Integrity) | **KEEP** | Removes "junk" rows (None, empty, 1-char, punct-only) that produce unusable vectors. |
| **2.8.2** (Replace URLs) | **KEEP** | Essential noise-reduction and privacy step ([URL]). |
| **2.8.4** (Replace Usernames) | **KEEP** | Essential for data anonymization and privacy ([USER]). |
| **2.8.6** (Consecutive Letters) | **KEEP** | Noise reduction (reduces 3+ repeats to 2) while preserving word structure. |
| **2.8.1** (Lowercasing) | **DISCARD** | LLMs rely on case sensitivity (e.g., distinguishing "Drug X" from general words). |
| **2.8.3** (Emojis/Emoticons) | **DISCARD** | Emojis carry valuable sentiment context for LLM interpretation. |
| **2.8.5** (Non-Alphabets) | **DISCARD** | Punctuation is critical for sentence chunking and structural meaning. |
| **2.8.7** (Short Words) | **DISCARD** | Essential for grammatical meaning (e.g., "no", "is", "a"). |
| **2.8.8** (Stopwords) | **DISCARD** | Necessary context for identifying subjects in side-effect reports. |

## Creating a supertext column for RAG

### Initial Checks in Fields

"During data exploration, we made a consistency check on DrugIds. It was found that 379 DrugIds were mapped to multiple distinct Drug names, indicating potential synonyms in the raw dataset."

Doing some research we found that for RAG, this is actually beneficial. There is no need to delete or "fix" these rows. Because transformer-based embedding models (like all-MiniLM-L6-v2),already understand that some drug names should  be the same i.e "Tylenol" and "Acetaminophen" are semantically similar.  CHECK

By keeping all possible names in the rag_block, we increase the retrieval span —the user's query will find relevant documents regardless of which name they use.

In [9]:
#find if I have multiple names with 1 drug ID. This is a step to construct the rag_text column
# Group by DrugId and get the unique list of Drug names for each
drug_mapping = df_rag.groupby('DrugId')['Drug'].unique()

# Filter to keep only DrugIds that map to more than 1 name
multiple_names = drug_mapping[drug_mapping.apply(len) > 1]

print(f"Found {len(multiple_names)} DrugIds with multiple names.")

# Display a few examples
if len(multiple_names) > 0:
    print("\nExamples of DrugIds with multiple names:\n")
    print(multiple_names.head(20))

Found 379 DrugIds with multiple names.

Examples of DrugIds with multiple names:

DrugId
18                                                            [salicylic acid shampoo, salicylic acid liquid, salicylic acid foam, salicylic acid film-forming liquid with applicator, salicylic acid gel]
73                                                                                                                                                          [loratadine tablet,disintegrating, loratadine]
128                                                                                                                          [itraconazole solution, itraconazole capsule, solid dispersion, itraconazole]
197                                                                                                                                                                          [sporanox solution, sporanox]
214                                                                                                

Another issue that we check was whether the Date field would be a problem since it is an object and not a date perse. The LLMs are clever enough to understand a date, even when it is a string with the date format

In [10]:
#Check the details of the data fields.
df_rag.info()

<class 'pandas.core.frame.DataFrame'>
Index: 319933 entries, 0 to 362805
Data columns (total 12 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   Age            319933 non-null  object
 1   Condition      319933 non-null  object
 2   Date           319933 non-null  object
 3   Drug           319933 non-null  object
 4   DrugId         319933 non-null  int64 
 5   EaseofUse      319933 non-null  int64 
 6   Effectiveness  319933 non-null  int64 
 7   Reviews        319933 non-null  object
 8   Satisfaction   319933 non-null  int64 
 9   Sex            319933 non-null  object
 10  Sides          319933 non-null  object
 11  UsefulCount    319933 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 31.7+ MB


### The RAG Block

Create a new supercolumn that will have all the informatin of the dataframe. We are naming each information with a label so the LLM will understand that this is Metadata, or it will be a Rating, or the actual review. This will be part of all the documents, so the LLM will be able to make this part of its retrieval understanding

In [47]:
# Function to create the RAG block using the existing string format
def create_rag_block(row):
    return (
        f"Metadata: "
        f"Date: {row['Date']}, "
        f"Drug: {row['Drug']} (ID: {row['DrugId']}), "
        f"Condition: {row['Condition']}, "
        f"Patient Age: {row['Age']}, "
        f"Gender: {row['Sex']}. "
        f"Ratings: Satisfaction {row['Satisfaction']}/5, "
        f"Effectiveness {row['Effectiveness']}/5, "
        f"Ease of Use {row['EaseofUse']}/5. "
        f"Helpfulness: {row['UsefulCount']} people found this review useful. "
        f"Side Effects: {row['Sides']}. "
        f"Review Content: {row['Reviews']}"
    )

In [48]:
df_rag['rag_block'] = df_rag.apply(create_rag_block, axis=1)

In [13]:
df_rag.head()

,Age,Condition,Date,Drug,DrugId,EaseofUse,Effectiveness,Reviews,Satisfaction,Sex,Sides,UsefulCount,rag_block
0,75 or over,Stuffy Nose,9/21/2014,25dph-7.5peh,146724,5,5,I'm a retired physician and of all the meds I have tried for my allergies (seasonal and not) - this one is the most effective for me. When I first began using this drug some years ago - tiredness as a problem but is not currently.,5,Male,"Drowsiness, dizziness , dry mouth /nose/throat, headache , upset stomach , constipation , or trouble sleeping may occur.",0,"Metadata: Date: 9/21/2014, Drug: 25dph-7.5peh (ID: 146724), Condition: Stuffy Nose, Patient Age: 75 or over, Gender: Male. Ratings: Satisfaction 5/5, Effectiveness 5/5, Ease of Use 5/5. Helpfulness: 0 people found this review useful. Side Effects: Drowsiness, dizziness , dry mouth /nose/throat, headache , upset stomach , constipation , or trouble sleeping may occur.. Review Content: I'm a retired physician and of all the meds I have tried for my allergies (seasonal and not) - this one is the most effective for me. When I first began using this drug some years ago - tiredness as a problem but is not currently."
1,25-34,Cold Symptoms,1/13/2011,25dph-7.5peh,146724,5,5,cleared me right up even with my throat hurting it went away after taking the medicine,5,Female,"Drowsiness, dizziness , dry mouth /nose/throat, headache , upset stomach , constipation , or trouble sleeping may occur.",1,"Metadata: Date: 1/13/2011, Drug: 25dph-7.5peh (ID: 146724), Condition: Cold Symptoms, Patient Age: 25-34, Gender: Female. Ratings: Satisfaction 5/5, Effectiveness 5/5, Ease of Use 5/5. Helpfulness: 1 people found this review useful. Side Effects: Drowsiness, dizziness , dry mouth /nose/throat, headache , upset stomach , constipation , or trouble sleeping may occur.. Review Content: cleared me right up even with my throat hurting it went away after taking the medicine"
2,65-74,Other,7/16/2012,warfarin (bulk) 100 % powder,144731,2,3,why did my PTINR go from a normal of 2.5 to over \n100?,3,Female,,0,"Metadata: Date: 7/16/2012, Drug: warfarin (bulk) 100 % powder (ID: 144731), Condition: Other, Patient Age: 65-74, Gender: Female. Ratings: Satisfaction 3/5, Effectiveness 3/5, Ease of Use 2/5. Helpfulness: 0 people found this review useful. Side Effects: . Review Content: why did my PTINR go from a normal of 2.5 to over \n100?"
3,75 or over,Other,9/23/2010,warfarin (bulk) 100 % powder,144731,2,2,FALLING AND DON'T REALISE IT,1,Female,,0,"Metadata: Date: 9/23/2010, Drug: warfarin (bulk) 100 % powder (ID: 144731), Condition: Other, Patient Age: 75 or over, Gender: Female. Ratings: Satisfaction 1/5, Effectiveness 2/5, Ease of Use 2/5. Helpfulness: 0 people found this review useful. Side Effects: . Review Content: FALLING AND DON'T REALISE IT"
4,35-44,Other,1/6/2009,warfarin (bulk) 100 % powder,144731,1,1,"My grandfather was prescribed this medication (Coumadin) to assist in blood thinning due to a heart and thyroid condition. His primary doctor was aware that he was on an aspirin regiment and still prescribed this medicine, it caused his blood to thin out to much and he ended up internally bleeding to death. If you are going to take this medicine please ask your doctors about possible side effects or drug interactions.",1,Male,,1,"Metadata: Date: 1/6/2009, Drug: warfarin (bulk) 100 % powder (ID: 144731), Condition: Other, Patient Age: 35-44, Gender: Male. Ratings: Satisfaction 1/5, Effectiveness 1/5, Ease of Use 1/5. Helpfulness: 1 people found this review useful. Side Effects: . Review Content: My grandfather was prescribed this medication (Coumadin) to assist in blood thinning due to a heart and thyroid condition. His primary doctor was aware that he was on an aspirin regiment and still prescribed this medicine, it caused his blood to thin out to much and he ended up internally bleeding to death. If you are going to take this medicine please ask your doctors about possible side effects or drug interactions."


# Which device to use to run the code > torch.cuda

In [49]:
if torch.cuda.device_count()>0:
    my_device = "cuda"
    print(f"You have {torch.cuda.device_count()} GPUs available.")
else:
    my_device = "cpu"
    print("You have no GPUs available. Running on CPU.")

You have 1 GPUs available.


# Embeddings Model

Model of choice

In [50]:
embeddings_model_name = 'BAAI/bge-small-en-v1.5'
#embeddings_model_name = 'sentence-transformers/all-MiniLM-L6-v2'

In [51]:
#Note: Because the model files are already in ./HF-CACHE, Hugging Face's library will automatically detect them locally and load them instantly without hitting the internet in a second run
embeddings_model = SentenceTransformer(embeddings_model_name,
                                       cache_folder="./HF-CACHE",
                                       device=my_device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# Large Language Model - Decoder

Download the LLM - predict the next token in a sequence given all preceding tokens. Causal LLM. It ingests the prompt containing both the **system **bold text** instructions, the retrieved database blocks (from your FAISS index)**, and the **user's explicit question**.  

Because it is a causal generator, it reads through the provided chunks autoregressively (token predicted is fed again into the input sequence to produce the next token) and crafts a fluent, natural-language response constrained strictly by the provided text boundaries.  

In [52]:
llm_model_name = "Qwen/Qwen3.5-0.8B"
#llm_model_name = "Qwen/Qwen3.5-0.8B"

In [53]:
tokenizer = AutoTokenizer.from_pretrained(llm_model_name, cache_dir="./HF-CACHE")
llm_model = AutoModelForCausalLM.from_pretrained(   llm_model_name,
                                                cache_dir="./HF-CACHE",
                                                device_map="auto",
                                                torch_dtype="auto")
#The pad_token is just a way to make all your data the same size so your GPU can process it efficiently.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Pad token was None, so it was set to eos token.")
    #As soon as the model predicts a single "token" (a word or part of a word), the streamer catches it, decodes it using the tokenizer you provided, and prints it to your output immediately.

#This is what creates the "typewriter" effect we see in tools like ChatGPT where the text appears to be "typing itself" out in real-time.
streamer = TextStreamer(tokenizer)

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

# Creating the Embeddings of the rag_block. (all info of dataframe)

Create the vectors / embeddings for the sentences in the rag_block of the datafreame. Then these will be saved in a vector database in order to compare with the embedding of the user question, and extract the most relevant response from the dataframe

This takes long to run, so after the creation of the vectors we save these embeddings in Google drive in order to load them again

In [19]:
# Extract the text chunks to be embedded
rag_documents = df_rag['rag_block'].tolist()

# Generate embeddings
# show_progress_bar=True helps you track if it's running
# batch_size=32 is a safe default to prevent memory crashes in Colab
my_embeddings = embeddings_model.encode(rag_documents, show_progress_bar=True, batch_size=32)

# Convert to float32, which is the format required by FAISS - Vector Database to be using later on.
my_embeddings = np.array(my_embeddings).astype('float32')

print(f"Embeddings shape: {my_embeddings.shape}")

Batches:   0%|          | 0/9998 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Save the review embeddings in Google Drive

In [ ]:
#Save the embeddings in case we need to run again , we can only load them.NPY is a binary file easy to be retrieved later on
# Save to your Google Drive folder (assuming ROOT_DATA_DIR is defined)
safe_model_name = embeddings_model_name.replace("/", "_")
file_name_embeddings = f"review_embeddings_{safe_model_name}.npy"

# Save the embeddings to Google Drive using the ROOTT_DATA_DIR
np.save(ROOT_DATA_DIR / file_name_embeddings, my_embeddings)

print(f"Embeddings saved to Google Drive as: {file_name_embeddings}")

In [54]:
#load Embeddings if needed
# Load the pre-computed embeddings
safe_model_name = embeddings_model_name.replace("/", "_")
file_name_embeddings = f"review_embeddings_{safe_model_name}.npy"
my_embeddings = np.load(ROOT_DATA_DIR /file_name_embeddings)
print(f"Embeddings created with {embeddings_model_name} model are loaded")

Embeddings created with BAAI/bge-small-en-v1.5 model are loaded


#Building Indexes for Vector Database

In [21]:
# Initialize and add to index
dimension = my_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(my_embeddings)

print(f"Index built with {index.ntotal} vectors.")


Index built with 319933 vectors.


### Save the review indexes of FAISS

In [29]:
# Save the FAISS index to your Google Drive
# Using your defined ROOT_DATA_DIR
safe_model_name = embeddings_model_name.replace("/", "_")
file_name_index = f"review_index_{safe_model_name}.faiss"
#
faiss.write_index(index, str(ROOT_DATA_DIR / file_name_index))
print(f"FAISS Indexes saved to Google Drive as: {file_name_index}")

FAISS Indexes saved to Google Drive as: review_index_BAAI_bge-small-en-v1.5.faiss


In [55]:
#Load the indexes if needed to be loaded

# Define the exact path where you saved it
safe_model_name = embeddings_model_name.replace("/", "_")
file_name_index = f"review_index_{safe_model_name}.faiss"
index_path = ROOT_DATA_DIR / file_name_index

# Check if the file exists before trying to load (good practice!)
if index_path.exists():
    # Read the index
    # Note: read_index requires a string, not a Path object
    index = faiss.read_index(str(index_path))
    print(f"Successfully loaded index with {index.ntotal} vectors.")
else:
    print(f"Error: The file {index_path} was not found. Check your file name or path.")

Successfully loaded index with 319933 vectors.



# RAG Process

In [58]:
#Parameters
k_mostrelevant = 30
streamer = None
MAXIMUM_TOKENS = 1024
#--------------------------------------------------------------------------------
#Create embeddings of the question adn retrieve the top k most relevant
#documents to the user question from the vector database
#

def get_context(query, embeddings_model, index, df, k=k_mostrelevant):
    """
    Retrieves the top k most relevant documents from the vector store.

    Args:
        query (str): The user's question.
        embeddings_model: Your loaded SentenceTransformer model.
        index: Your FAISS index object.
        df (pd.DataFrame): Your main dataset (df_rag).
        k (int): Number of chunks to retrieve.

    Returns:
        list: A list of text chunks (rag_block) containing the context.
    """
    # Embed the user query
    # The model expects a list, so we wrap the query in []
    query_vector = embeddings_model.encode([query])

    # FAISS strictly requires float32 data types
    query_vector = np.array(query_vector).astype('float32')

    # Search the index
    # distances: how similar the result is (lower is better for L2 distance)
    # indices: the row numbers in your dataframe
    distances, indices = index.search(query_vector, k)

    # Retrieve the context
    # indices[0] is the list of results for our 1 query
    # We use .iloc to extract these specific rows from the dataframe
    #retrieved_texts = df_rag.iloc[indices[0]]['rag_block'].tolist()
    retrieved_texts = df_rag.iloc[indices[0]]['rag_block'].unique().tolist() # added the unique since many times the same reviews are listed in the tests!

    return retrieved_texts
#-------------------------------------------------------------------------------
#  Settup the messages
#
def create_messages(user_question, context_list):
    # Combine the list of retrieved context chunks into one string
    context_str = "\n\n".join(context_list)

    system_instruction = (
        "You are a medical research assistant. "
        "Use the provided <context> (which includes patient reviews and ratings such as EaseofUse, Effectiveness and Satisfaction) to answer the <question>. "
        "Prioritize answers with larger UsefulCount. "
        "Do not make up medical facts outside of the provided context. "
        "Do not use outside knowledge.\n\n"
        "Instructions:\n"
        "- If the question asks about 'side effects', prioritize the 'Side Effects' field.\n"
        "- If the question asks about 'effectiveness' or comparative drug performance, prioritize the numerical 'Effectiveness' ratings over subjective text sentiment.\n"
        "- If the question asks about 'satisfaction', prioritize the numerical 'Satisfaction' ratings over subjective text sentiment.\n"
        "- If the question asks about 'ease of use', prioritize the numerical 'EaseofUse' ratings and specific usability comments in the reviews.\n"
        "- When performing drug comparisons, prioritize numerical ratings across the metadata rather than selective review quotes.\n"
        "- When summarizing data or trends from the snippets, synthesize the patterns directly from the retrieved text rather than refusing due to a lack of global data.\n"
        "- Do not invent rating scales (like 6-10) if the ratings are out of 5.\n"
        "- When answering, directly address the user's prompt using the patterns, counts, and examples found within the retrieved text snippets.\n"
        "- If a specific data point is missing from the snippets, concisely state what the retrieved snippets show rather than issuing a blanket refusal.\n"
        "- Answer directly and concisely based on the text provided."
    )
    # Construct the message structure
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {user_question}"}
    ]
    return messages
#-------------------------------------------------------------------------------
# From the LLM Model selected generate the response
#
def generate_response(messages, tokenizer, llm_model, streamer):
    #MAXIMUM_TOKENS = 1024
    # Apply Qwen's specific chat template for the LLM
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize the input and move it to the same device as the model (GPU)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    attention_mask = inputs["attention_mask"]

    #print("AI Response: ")

    # Create a GenerationConfig object to safely pass parameters like temperature
    gen_config = GenerationConfig(
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=MAXIMUM_TOKENS,
        temperature=0.7,
        do_sample=True, # Required when using temperature sampling
    )

    # 4. Wrap generation in torch.no_grad() to drastically cut VRAM usage and prevent OOM
    with torch.no_grad():
        outputs = llm_model.generate(
            inputs["input_ids"],
            streamer=streamer,
            attention_mask=attention_mask,
            generation_config=gen_config
        )
    return outputs
#
#--------------------------------------------------------------------------
#Chatbot loops
# Ask the Chatbot function, using RAG as above
def ask_chatbot(question):
    context = get_context(question, embeddings_model, index, df_rag, k=k_mostrelevant)
    messages = create_messages(question, context)
    outputs = generate_response(messages, tokenizer, llm_model,streamer)
    return outputs

# Ensure your cleaning function is defined
def get_clean_answer(outputs, tokenizer):
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    clean_text = re.sub(r'<think>.*?</think>', '', full_text, flags=re.DOTALL)
    clean_text = clean_text.strip()
    if "assistant" in clean_text:
        clean_text = clean_text.split("assistant")[-1].strip()
    return clean_text, full_text


In [57]:
# Test the retrieval retrieval
test_query = "What are the side effects of Lisinopril?"
context = get_context(test_query, embeddings_model, index, df_rag, k=k_mostrelevant)

# Print the results to verify
for i, text in enumerate(context):
    print(f"--- Context {i+1} ---\n{text}\n")

--- Context 1 ---
Metadata: Date: 2/28/2009, Drug: lisinopril (ID: 6873), Condition: High Blood Pressure, Patient Age: 55-64, Gender: Male. Ratings: Satisfaction 1/5, Effectiveness 3/5, Ease of Use 3/5. Helpfulness: 10 people found this review useful. Side Effects: Dizziness ,  lightheadedness , tiredness, or  headache  may occur as your body adjusts to the  medication . Dry  cough  may also occur.. Review Content: Not sure which of these is relevant, but on the two weeks I've been on the drug I've noticed the following: light-headedness, throbbing at the back of my head (left side), a fever blister,loud ringing in the ears at night, sharp,temporary pain in left kidney, itchiness on right forearm, cramps in left foot, blue zig-zag streak several times at night with my eyes closed, slight nausea for several days.

--- Context 2 ---
Metadata: Date: 10/25/2011, Drug: lisinopril (ID: 6873), Condition: High Blood Pressure, Patient Age: 75 or over, Gender: Female. Ratings: Satisfaction 1/5, 

# Test Chatbot

## Question

In [59]:
# Execute a question to generate 'outputs'
question_to_answer = "What are the side effects of Lisinopril?"
outputs = ask_chatbot(question_to_answer)

# Get the answers in cleaned and raw mode
clean_answer, raw_answer = get_clean_answer(outputs, tokenizer)

## Printouts

In [60]:
print(f"\n--- Testing Question: {question_to_answer} ---")
# Print with clean formatting and visual separation
print("=" * 60)
print("📌 RAW MODEL OUTPUT (with special tokens/prompts):")
print("=" * 60)
print(raw_answer)

print("\n" + "=" * 60)
print("✨ CLEANED ANSWER (ready for the user/UI):")
print("=" * 60)
print(clean_answer)
print("=" * 60)


--- Testing Question: What are the side effects of Lisinopril? ---
📌 RAW MODEL OUTPUT (with special tokens/prompts):
system
You are a medical research assistant. Use the provided <context> (which includes patient reviews and ratings such as EaseofUse, Effectiveness and Satisfaction) to answer the <question>. Prioritize answers with larger UsefulCount. Do not make up medical facts outside of the provided context. Do not use outside knowledge.

Instructions:
- If the question asks about 'side effects', prioritize the 'Side Effects' field.
- If the question asks about 'effectiveness' or comparative drug performance, prioritize the numerical 'Effectiveness' ratings over subjective text sentiment.
- If the question asks about 'satisfaction', prioritize the numerical 'Satisfaction' ratings over subjective text sentiment.
- If the question asks about 'ease of use', prioritize the numerical 'EaseofUse' ratings and specific usability comments in the reviews.
- When performing drug comparisons,

#Prompt Engineering

Since our dataset (df_rag) contains specific fields like Sides, Effectiveness, and Satisfaction and other metadata, our prompt engineering should guide the model to prioritize these fields based on what the user is asking.

Index(['Age', 'Condition', 'Date', 'Drug', 'DrugId', 'EaseofUse',
       'Effectiveness', 'Reviews', 'Satisfaction', 'Sex', 'Sides',
       'UsefulCount'],
      dtype='object')

## Test Questions


In [ ]:
test_questions = [
    "What are the side effects of Lisinopril?",           # Target: Side Effects
    "How effective is Lisinopril for blood pressure?",    # Target: Effectiveness
    "Are people generally satisfied with Lisinopril?",    # Target: Satisfaction
    "Can I take Lisinopril for the common cold?",         # Target: Unknown/Context check
    "What is the age of people mostly taking Lisinopril?",
    "How has the satisfaction for Lisinopril evolved through the years?",
    "How has the ease of use for Lisinopril evolved through the years?"
]
for q in test_questions:
    print(f"\n--- Testing: {q} ---")
    outputs = ask_chatbot(q)

    # Unpack both values returned from the function
    clean_answer, raw_answer = get_clean_answer(outputs, tokenizer)

    #print("\n[RAW]:")
    #print(raw_answer)

    print("\n[CLEANED]:")
    print(clean_answer)


--- Testing: What are the side effects of Lisinopril? ---

[CLEANED]:
Based on the provided context, the side effects of Lisinopril are listed under the "Side Effects" field for all entries in the metadata, though the specific details vary slightly by patient.

The common side effects mentioned in every review are:
*   Dizziness
*   Lightheadedness
*   Tiredness
*   Headache

Additional side effects noted by specific patients include:
*   Dry cough
*   Itchiness (itchiness on the right forearm)
*   Cramps
*   Blue zig-zag streaks in the eyes
*   Mild nausea
*   Sharp, temporary pain in the left kidney
*   Fatigue
*   Muscle and joint pain/swelling
*   Weakness
*   Tiredness (mentioned frequently)
*   General sick feeling
*   Thirsty
*   Scalp itching
*   Body aches
*   Loss of appetite
*   Headaches (often described as throbbing)
*   Other allergic reactions such as hives, skin scaling/shingles, chills, and hot flashes.

**Note:** The text indicates that these side effects may occur a

##Questions for Prompt Engineering

Create a fuction for testing multiple questions in a list

In [132]:
def run_evaluation(questions_list):
    """
    Runs a list of test questions through the chatbot, extracts
    both cleaned and raw answers, and prints them with clean formatting.
    """
    for q in questions_list:

        # Ask the chatbot to get the generated token outputs
        outputs = ask_chatbot(q)

        # 2. Unpack both values returned from your cleaning function
        clean_answer, raw_answer = get_clean_answer(outputs, tokenizer)

        # 3. Print the cleaned answer
        print(f"\n" + "=" * 60)
        print(f"🟢  Question: {q} ---")
        print("\n" + "=" * 60)
        print("✨ CLEANED ANSWER: ---")
        print("=" * 60)
        print(clean_answer)
        print("=" * 60)

###Factual Questions

These questions idendify whether the llm model accurately pulls specific facts (like Side Effects and Satisfaction ratings).


In [133]:
questions_list = [
    "What are the reported side effects for Lisinopril?",
    "What is the average satisfaction rating for patients using Lisinopril for High Blood Pressure?"
    ]
run_evaluation(questions_list)


🟢  Question: What are the reported side effects for Lisinopril? ---

✨ CLEANED ANSWER: ---
Based on the provided context, the reported side effects for Lisinopril include a wide range of symptoms:

*   **Dizziness**
*   **lightheadedness**
*   **Headache**
*   **Tiredness**
*   **Nausea**
*   **Bloating/Gas**
*   **Constipation**
*   **Cold feet**
*   **Muscle aches**
*   **Fatigue**
*   **Weakness**
*   **Depression**
*   **Hair loss**
*   **Flu-like symptoms** (fever, chills, hot flashes, etc.)
*   **Swelling**
*   **Allergic reactions** (rash, hives, swelling, itching)
*   **Coughing**
*   **Vomiting**

*Note: While these are consistently mentioned in the reviews, the context indicates that the symptoms occur as the body adjusts to the medication and may be temporary or severe in severity.*

🟢  Question: What is the average satisfaction rating for patients using Lisinopril for High Blood Pressure? ---

✨ CLEANED ANSWER: ---
Based on the provided context, here is the analysis of the

#### Answers through Pandas

In [134]:
# Filter dataframe for Lisinopril and High Blood Pressure
drug_filter = df_rag['Drug'].str.contains('lisinopril', case=False, na=False)
condition_filter = df_rag['Condition'].str.contains('High Blood Pressure', case=False, na=False)
subset_df = df_rag[drug_filter & condition_filter]

# Check ground truth for Q1.1 (Side Effects)
print("--- Ground Truth: Side Effects Snippets ---")
print(subset_df['Sides'].dropna().unique()[:k_mostrelevant])

# Check ground truth for Q1.2 (Average Satisfaction)
avg_satisfaction = subset_df['Satisfaction'].mean()
print(f"\n--- Ground Truth: Average Satisfaction --- {avg_satisfaction:.2f}/5")

--- Ground Truth: Side Effects Snippets ---
['Dizziness ,  lightheadedness , tiredness, or  headache  may occur as your body adjusts to the  medication . Dry  cough  may also occur.']

--- Ground Truth: Average Satisfaction --- 2.50/5


### Comparative - Reasoning Questions


These are difficult questions requiring the system to retrieve and compare multiple entries or filter by metadata constraints like Age.

In [145]:
questions_list = [
    "Which drug is generally considered more effective for Depression: Cymbalta or Lexapro?",
    "Do older patients (75 or over) report different side effects for Lisinopril than younger patients (25-34)?"
    ]
run_evaluation(questions_list)


🟢  Question: Which drug is generally considered more effective for Depression: Cymbalta or Lexapro? ---

✨ CLEANED ANSWER: ---
Based on the provided reviews, **Cymbalta** is generally considered more effective for depression compared to Lexapro.

Here is the evidence supporting this conclusion:

*   **High Effectiveness Ratings:**
    *   Multiple reviews explicitly state that Cymbalta has "been a fairly positive help for my depression" (Review #12/31/2007, Review #4/27/2009, Review #9/27/2007).
    *   Many reviews note that Cymbalta has "helped more than most others" (Review #12/31/2007, Review #4/1/2010).
    *   A single review (#11/3/2008) notes Cymbalta "helped with anxiety and depression" and "Much, Much better with less side effects then, Lexapro," implying Cymbalta is superior in efficacy.
    *   Reviews #4/1/2010 and #11/17/2007 explicitly state Cymbalta has "far more effective than most others" and has "no negative side effects."

*   **Comparative Effectiveness:**
    *  

#### Answers though Pandas

In [146]:
# Ground truth check for Q2.1 (Comparing Effectiveness of two drugs for Depression)
drugs_to_compare = ['cymbalta', 'lexapro']
comp_df = df_rag[df_rag['Drug'].str.lower().isin(drugs_to_compare) & df_rag['Condition'].str.contains('Depression', case=False, na=False)]
print("--- Ground Truth: Average Effectiveness Comparison ---")
print(comp_df.groupby('Drug')['Effectiveness'].mean())

# Ground truth check for Q2.2 (Comparing Side Effects by Age Group for Lisinopril)
lisinopril_df = df_rag[df_rag['Drug'].str.contains('lisinopril', case=False, na=False)]
print("\n--- Ground Truth: Side Effects for Age 25-34 ---")
print(lisinopril_df[lisinopril_df['Age'] == '25-34']['Sides'].dropna().unique()[:k_mostrelevant])

print("\n--- Ground Truth: Side Effects for Age 75 or over ---")
print(lisinopril_df[lisinopril_df['Age'] == '75 or over']['Sides'].dropna().unique()[:k_mostrelevant])

--- Ground Truth: Average Effectiveness Comparison ---
Drug
cymbalta    3.384615
lexapro     3.608463
Name: Effectiveness, dtype: float64

--- Ground Truth: Side Effects for Age 25-34 ---
['Dizziness ,  lightheadedness , tiredness, or  headache  may occur as your body adjusts to the  medication . Dry  cough  may also occur.']

--- Ground Truth: Side Effects for Age 75 or over ---
['Dizziness ,  lightheadedness , tiredness, or  headache  may occur as your body adjusts to the  medication . Dry  cough  may also occur.'
 ' ']


### Summary - Inference  Questions

These test the LLM's ability to synthesize narrative text from the Reviews column and link subjective text context to low/high ratings.

In [159]:
questions_list = [
    "Can you summarize the patient experience for Metformin regarding Type 2 Diabetes?",
    "Why might a patient give Metformin a low satisfaction score (1/5)?"
    ]
run_evaluation(questions_list)


🟢  Question: Can you summarize the patient experience for Metformin regarding Type 2 Diabetes? ---

✨ CLEANED ANSWER: ---
Based on the provided review snippets, here is a summary of the patient experience for Metformin regarding Type 2 Diabetes:

**Common Side Effects**
The majority of the reviews consistently highlight gastrointestinal (GI) side effects, specifically nausea, vomiting, stomach upset, diarrhea, and metallic taste in the mouth. Some patients also report weakness (often described as lethargy) and headaches.

**Patient Perspective**
Patients report significant positive experiences, frequently describing themselves as feeling "brand new" and experiencing "no complications." Several reviews highlight substantial improvements in their condition:
*   **Satisfaction:** Most reviews show perfect ratings (5/5) across Satisfaction, Effectiveness, and Ease of Use.
*   **Blood Sugar Control:** Many patients report their blood sugar levels dropping significantly, with one patient no

#### Answers through Pandas

In [160]:
# Filter for Metformin and Type 2 Diabetes
met_df = df_rag[df_rag['Drug'].str.contains('metformin', case=False, na=False) & df_rag['Condition'].str.contains('Diabetes', case=False, na=False) & df_rag['Condition'].str.contains('Type 2', case=False, na=False)]

# Ground truth check for Q3.1 (General reviews summary)
print("--- Ground Truth: Sample Review Text for Metformin ---")
for rev in met_df['Reviews'].dropna().head(k_mostrelevant):
    print(f"- {rev}\n")

# Ground truth check for Question : Why low satisfaction scores occur
print("--- Ground Truth: Reviews with 1/5 Satisfaction ---")
low_sat_reviews = met_df[met_df['Satisfaction'] == 1]['Reviews'].dropna()
for rev in low_sat_reviews.head(3):
    print(f"- {rev}\n")

--- Ground Truth: Sample Review Text for Metformin ---
- It gave me non-stop diarrhea and severe stomach cramps.  Dr lowered dose and the symptoms still persisted aggressively.  Felt like I had severe flu all the time.  I quit taking it and Dr put me on something else that works much better.

- Seems to control blood sugar erratically

- It gave me non-stop diarrhea and severe stomach cramps.  Dr lowered dose and the symptoms still persisted aggressively.  Felt like I had severe flu all the time.  I quit taking it and Dr put me on something else that works much better.

- Seems to control blood sugar erratically

- I have been taking this drug for 4 months now. Before I started taking it, I read all of the reviews about the drug. Several people have commented that diarrhea was a symptom that they noticed. I put myself on a keto diet, restricting my carbs...especially sugar...and I had no symptoms of diarrhea. Until, that is, yesterday. Super bowl Sunday!!! My wife had made some fudge f

# RAG Evaluation & Troubleshooting Report

Below is the evaluation summary tracking tge prompt engineering iterations, retriever adjustments (most relevan reviews scaling), and accuracy checks against Pandas same selections
---

## Factual, Comparative, and Summary Questions Evaluation

| Question & Phase | LLM Output | Ground Truth (Pandas) | Correction & System Prompt Tuning / Analysis |
| :--- | :--- | :--- | :--- |
| **1. What are the reported side effects for Lisinopril?** | Divided side effects into fabricated "Ratings 1-5" and "Ratings 6-10" groups, listing common side effects alongside hallucinated severe reactions. | ['Dizziness , lightheadedness , tiredness, or headache may occur as your body adjusts to the medication. Dry cough may also occur.'] | Enforce strict grounding rules in system_instructions to prevent the model from inventing non-existent scales.<br><br>- Do not invent rating scales (like 6-10) if the ratings are out of 5. |
| *-- After Correction --* | *Successfully compiled a clean, comprehensive bulleted list of side effects grounded strictly in the text snippets, completely dropping the fabricated scales.* | *[Same]* | *Prompt rules successfully eliminated scale fabrication.* |
| **2. What is the average satisfaction rating for patients using Lisinopril for High Blood Pressure?** | Counted 9 reviews with 5/5 ratings, incorrectly evaluation the average satisfaction and concluded an average satisfaction of 0.56 / 5. | 2.50 / 5 (across the filtered subset with Panddas) | Instruct the model (system_instructions) that calculations shouldn't use arbitrary division and that insights apply only to retrieved chunks.<br><br>- When calculating averages... do not perform arbitrary division.<br>- Explicitly state that insights are based only on retrieved review snippets. |
| *-- After Correction --* | *Recognized its limitation instead of making up flawed arithmetic, explicitly stated that the context only provides individual chunk metadata, and identified the local range (3/5 to 5/5).* | *[Same]* | *Prompt rules successfully halted math hallucinations and clarified response scope.* |
| **3. Which drug is generally considered more effective for Depression: Cymbalta or Lexapro?** | Concluded that Cymbalta is more effective based on individual patient review highlights and switching testimonials. | cymbalta: 3.38 / 5<br>lexapro: 3.60 / 5 (Lexapro holds a higher average effectiveness score). | ⚠️ **Persistent Discrepancy:** The model favored subjective review text over numerical metadata.<br><br>*Fix:* Instructed model (system_instructions) to prioritize numerical Effectiveness ratings over text sentiment during comparisons. |
| *-- After Correction --* | *Answer remained the same (still favored Cymbalta due to the qualitative testimonials clustered in the retrieved chunk sample).* | *[Same]* | *Demonstrates that text-heavy retrieval can sometimes override prompt instructions if glowing text reviews heavily outweigh metadata numbers in top- retrieved reviews.* |
| **4. Do older patients (75 or over) report different side effects for Lisinopril than younger patients (25-34)?** | Stated that the context lacked a direct comparative demographic breakdown and initially refused to make a comparison. | Age 25-34: Standardized side effect text.<br>Age 75+: Standardized side effect text (plus empty strings). | * Add additional system_instructins: If the question asks about 'side effects', prioritize the 'Side Effects' field*|
| *-- After Correction --* | *Successfully analyzed text fields across both age groups, recognized reported side effects were identical, and answered "No" with full factual alignment.* | *[Same]* | System instructions can make a bit difference in the model performance |
| **5. Can you summarize the patient experience for Metformin regarding Type 2 Diabetes?** | Provided a structured summary noting satisfaction is mostly 5/5, side effects include nausea/diarrhea/stomach upset, and effective A1C control over time. | A mix of reviews ranging from severe GI issues (non-stop diarrhea, cramps, ER visits) to positive ones (blood sugar control, weight loss). | 🟡 **Mixed Accuracy / Sample Bias:** Moving $k$ to 30 resolved previous OOM crashes, but the retriever pulled a sample leaning toward positive testimonials, glossing over severe debilitating cramps. |
| *-- After Correction ($k=30$) --* | *Ran smoothly without crashing (solving OOM), but exhibited sample bias by leaning into positive testimonials found in the top-30 chunk sample.* | *[Same]* | *Highlights the inherent sampling limits of vector retrievers.* |
| **6. Why might a patient give Metformin a low satisfaction score (1/5)?** | Listed intestinal issues, daily dizziness, blood sugar dropping too low/fluctuating, cholesterol spikes, and anxiety. | **Actual 1/5 Review Texts:**<br>1. Non-stop diarrhea, severe stomach cramps, flu-like feeling.<br>2. Violent explosive "moaning" diarrhea with massive cramps. | ⚠️ **Context Mismatch in Retrieval:** The LLM accurately summarized the retrieved top-30 chunks, but those chunks captured a diverse spread of complaints (dizziness, sugar spikes) rather than the database-wide 1/5 truth (heavily dominated by acute GI distress). |
| *-- After Correction ($k=30$) --* | *Accurately synthesized the specific low-rated reviews present in the retrieved sample, though they reflected a wider variety of complaints than the database-wide baseline.* | *[Same]* | *Reinforces that an LLM's grounding is strictly bound to what the retriever surfaces.* |

---

## Key Takeaways

* **The LLM is only as good as its retriever:** Increasing $k$ from 20 to 30  narrative synthesis, but exposed sampling variance where top-$k$ chunks don't always mirror database-wide aggregate metrics.
* **Metadata Enforcement:** System prompt constraints successfully eliminated math hallucinations (like dividing 5 by 9) and forced the model to acknowledge its boundaries as a document-based assistant.

#RAGAS Framework (Retrieval Augmented Generation Assessment)

Create a dataset which holds the results of the RAG implementation. This will then be fed to RAGAS to evaluate


In [61]:
# 1. Define your test questions
questions = [
    "What are the reported side effects for Lisinopril?",
    "What is the average satisfaction rating for patients using Lisinopril for High Blood Pressure?",
    "Which drug is generally considered more effective for Depression: Cymbalta or Lexapro?",
    "Do older patients (75 or over) report different side effects for Lisinopril than younger patients (25-34)?",
    "Can you summarize the patient experience for Metformin regarding Type 2 Diabetes?",
    "Why might a patient give Metformin a low satisfaction score (1/5)?",
    ]


# 2. Define optional reference ground truths
#references = [
 #   "Dizziness, headache, and dry cough.",
#   "Yes, effective for A1C control with common GI side effects."
#]

user_inputs = []
responses = []
retrieved_contexts = []

# 3. Loop through your questions to gather context and generated responses
for q in questions:
    # Retrieve context chunks from FAISS using your existing function
    context = get_context(q, embeddings_model, index, df_rag, k=k_mostrelevant)

    # Generate the response using your chatbot pipeline
    outputs = ask_chatbot(q)

    # Clean the response text using your function
    clean_answer, _ = get_clean_answer(outputs, tokenizer)

    # Store them in lists
    user_inputs.append(q)
    responses.append(clean_answer)
    retrieved_contexts.append(context)

# 4. Assemble the evaluation dictionary
eval_data = {
    "user_input": user_inputs,
    "response": responses,
    "retrieved_contexts": retrieved_contexts #,
   # "reference": references
}

# 5. Create the Hugging Face Dataset object for Ragas
dataset = Dataset.from_dict(eval_data)

print(f"Successfully gathered {len(dataset)} evaluation samples ready for Ragas!")

Successfully gathered 6 evaluation samples ready for Ragas!


In [ ]:
Save the RAGAS dataset in shared Google Drive

In [62]:
# Define path in your shared Google Drive folder
drive_dataset_path = ROOT_DATA_DIR / "my_rag_eval_dataset"

# Save dataset
dataset.save_to_disk(str(drive_dataset_path))
print(f"Dataset saved securely to Google Drive at: {drive_dataset_path}")

Saving the dataset (0/1 shards):   0%|          | 0/6 [00:00<?, ? examples/s]

Dataset saved securely to Google Drive at: /content/drive/MyDrive/nlp_project_data/my_rag_eval_dataset


In [11]:
#Read the dataset produced for importing into RAGAS from shared Google Drive
drive_dataset_path = ROOT_DATA_DIR / "my_rag_eval_dataset"
dataset = load_from_disk(str(drive_dataset_path))
print("Dataset loaded from Google Drive successfully!")

Dataset loaded from Google Drive successfully!


In [84]:
import os
from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import faithfulness, answer_relevancy
from ragas import evaluate

# ==========================================
# 1. SECURELY LOAD API KEY
# ==========================================
# If using a .env file or local text file, load it here:
# with open("secret_key.txt", "r") as f:
#     os.environ["OPENAI_API_KEY"] = f.read().strip()

# Or ensure it's already set in your session environment
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


# ==========================================
# 2. INITIALIZE MODELS (WITH WRAPPERS & TOKENS)
# ==========================================
# Increase max_tokens to prevent "IncompleteOutputException" on long responses
raw_llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=2048)
raw_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Wrap them using Ragas wrappers to ensure compatibility and prevent attribute errors
eval_llm = LangchainLLMWrapper(raw_llm)
eval_embeddings = LangchainEmbeddingsWrapper(raw_embeddings)


# ==========================================
# 3. RUN THE RAGAS EVALUATION
# ==========================================
print("Starting RAGAS evaluation pipeline...")

result = evaluate(
    dataset=dataset,  # Ensure your 'dataset' variable (EvaluationDataset) is defined above
    metrics=[faithfulness, answer_relevancy],
    embeddings=eval_embeddings,
    llm=eval_llm,
)

# Convert results into a clean Pandas DataFrame
eval_df = result.to_pandas()
display(eval_df)



/tmp/ipykernel_111384/3119381851.py:6: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_111384/3119381851.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_111384/3119381851.py:28: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(raw_llm)
/tmp/ipykernel_111384/3119381851.py:29

Starting RAGAS evaluation pipeline...


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()


user_input  \
0                                                          What are the reported side effects for Lisinopril?   
1              What is the average satisfaction rating for patients using Lisinopril for High Blood Pressure?   
2                      Which drug is generally considered more effective for Depression: Cymbalta or Lexapro?   
3  Do older patients (75 or over) report different side effects for Lisinopril than younger patients (25-34)?   
4                           Can you summarize the patient experience for Metformin regarding Type 2 Diabetes?   
5                                          Why might a patient give Metformin a low satisfaction score (1/5)?   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [88]:
# ==========================================
# 4. SAVE RESULTS TO GOOGLE DRIVE SHARED FOLDER
# ==========================================
# Assumes ROOT_DATA_DIR is already defined in your notebook setup
try:
    output_file_path = ROOT_DATA_DIR / "ragas_evaluation_results.xlsx"
    eval_df.to_excel(output_file_path, index=False)
    print(f"✅ Evaluation results successfully saved to: {output_file_path}")
except NameError:
    # Fallback if ROOT_DATA_DIR isn't defined
    eval_df.to_excel("ragas_evaluation_results.xlsx", index=False)
    print("✅ Evaluation results saved locally as 'ragas_evaluation_results.xlsx'")


✅ Evaluation results successfully saved to: /content/drive/MyDrive/nlp_project_data/ragas_evaluation_results.xlsx


In [89]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=eval_df)

https://docs.google.com/spreadsheets/d/16onU5ar7tctuslhq6h30ls-7WSPL4LDSn1bqVROALMI/edit#gid=0


Save the evaluation excel in Google Drive

In [28]:
# Define your file path in your shared Google Drive directory
output_file_path = ROOT_DATA_DIR / "ragas_evaluation_results.xlsx"

# Export the DataFrame directly to Excel
eval_df.to_excel(output_file_path, index=False)

print(f"Evaluation results successfully saved to Google Drive at: {output_file_path}")

Evaluation results successfully saved to Google Drive at: /content/drive/MyDrive/nlp_project_data/ragas_evaluation_results.xlsx


#RAGAS Conclusion
Changing the instructions successfully fixed the Answer Relevancy bottleneck, transforming the system from an overy-causious assistant that refused to answer into an engaged responder.  
 However, it introduced a minor penalty in Faithfulness, proving the classic RAG engineering trade-off: making an LLM more helpful can slightly increase its tendency to drift from strict source bounds.